# Local Flask Runner

Use this notebook when running the project locally in VS Code/Jupyter, not in Google Colab.

Before using the app, make sure these trained checkpoint files exist:

- `checkpoints/ham10000_dcgan.pth`
- `checkpoints/skin_classifier.pth`

If they were trained in Colab, download/copy them from Drive into the local `checkpoints/` folder.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
print('Project root:', PROJECT_ROOT)
print('Python executable:', sys.executable)
print('Python version:', sys.version)

required_files = ['frontend/app.py', 'classifier.py', 'gan.py', 'data.py']
for path in required_files:
    print(path, 'OK' if Path(path).exists() else 'MISSING')

In [ ]:
from pathlib import Path
import shutil
import zipfile

project_checkpoints = Path('checkpoints')
project_checkpoints.mkdir(exist_ok=True)

search_roots = [
    Path.home() / 'Downloads',
    Path.home() / 'Desktop',
    Path.cwd(),
]

# First, look for direct .pth files.
for root in search_roots:
    if not root.exists():
        continue
    for name in ['ham10000_dcgan.pth', 'skin_classifier.pth']:
        matches = sorted(root.rglob(name), key=lambda p: p.stat().st_mtime, reverse=True)
        if matches:
            src = matches[0]
            dst = project_checkpoints / name
            if src.resolve() != dst.resolve():
                shutil.copy2(src, dst)
                print('Copied:', src, '->', dst)

# Then, look for the zip created by the Colab notebook.
if not all((project_checkpoints / name).exists() for name in ['ham10000_dcgan.pth', 'skin_classifier.pth']):
    zip_matches = []
    for root in search_roots:
        if root.exists():
            zip_matches.extend(root.rglob('medical_gan_checkpoints*.zip'))
            zip_matches.extend(root.rglob('*checkpoints*.zip'))
    zip_matches = sorted(set(zip_matches), key=lambda p: p.stat().st_mtime, reverse=True)
    if zip_matches:
        zip_path = zip_matches[0]
        print('Extracting:', zip_path)
        with zipfile.ZipFile(zip_path) as archive:
            archive.extractall(project_checkpoints)

for name in ['ham10000_dcgan.pth', 'skin_classifier.pth']:
    path = project_checkpoints / name
    print(path, 'OK' if path.exists() else 'MISSING')


## Install Local Dependencies

Run this once for the selected notebook kernel. CPU-only PyTorch is used because it is lighter and enough for running Flask locally.

In [ ]:
import sys

!{sys.executable} -m pip install -q flask pandas numpy matplotlib scikit-learn kaggle pillow
!{sys.executable} -m pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cpu

## Check Model Files

The app can open without checkpoints, but classification and generation need the `.pth` files below.

In [ ]:
from pathlib import Path

Path('checkpoints').mkdir(exist_ok=True)

required_checkpoints = [
    Path('checkpoints/ham10000_dcgan.pth'),
    Path('checkpoints/skin_classifier.pth'),
]

missing = []
for path in required_checkpoints:
    if path.exists():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f'{path}: OK ({size_mb:.1f} MB)')
    else:
        print(f'{path}: MISSING')
        missing.append(path)

if missing:
    print('\nCopy these files into the checkpoints folder before using the model actions:')
    for path in missing:
        print(' -', path)
else:
    print('\nAll checkpoints are ready.')

## Start Flask Locally

Run this cell, then open the printed local links in your browser. The server keeps running in the background.

In [ ]:
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

PORT = 5000
HOST = '127.0.0.1'

# Stop the previous server started by this notebook, if it exists.
import time

try:
    if flask_server.poll() is None:
        flask_server.terminate()
        time.sleep(1)
except NameError:
    pass

log_file = open('flask_local.log', 'w')
flask_server = subprocess.Popen(
    [sys.executable, '-m', 'flask', '--app', 'frontend/app.py', 'run', '--host', HOST, '--port', str(PORT)],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
)

time.sleep(4)
print('Server process id:', flask_server.pid)
print('Server return code:', flask_server.poll())

if flask_server.poll() is not None:
    print(Path('flask_local.log').read_text()[-4000:])
    raise RuntimeError('Flask did not start.')

test_url = f'http://{HOST}:{PORT}/classify'
response = urllib.request.urlopen(test_url, timeout=5)
print('Local test:', response.status, response.reason)

print('\nOpen these links:')
print('Classifier:', f'http://{HOST}:{PORT}/classify')
print('Generate:', f'http://{HOST}:{PORT}/generate')

## Stop Flask

Run this when you are done using the local website.

In [ ]:
try:
    flask_server.terminate()
    time.sleep(1)
    print('Flask stopped.')
except NameError:
    print('No Flask server variable found in this notebook session.')